# Inventory Discrepancy Analysis

In this notebook, we are going to review the discrepancies between two major sources for housing inventory data, Zillow and Realtor.com. We are going to map their region codes to a universal standard, format the dates and data, and align the two series for discrepancy analysis

In [1]:
import calendar
from datetime import datetime

import pandas as pd

In [2]:
# get source datasets
# Zillow overview pane: https://www.zillow.com/research/data/
# Realtor overview pane: https://www.realtor.com/research/data/
zillow_url = 'https://files.zillowstatic.com/research/public_csvs/invt_fs/Metro_invt_fs_uc_sfrcondo_month.csv?t=1684107340'
realtor_url = 'https://econdata.s3-us-west-2.amazonaws.com/Reports/Core/RDC_Inventory_Core_Metrics_Metro_History.csv'

zillow = pd.read_csv(zillow_url)
realtor = pd.read_csv(realtor_url)

/var/folders/rv/222w1rcx2035v5jbm6501tvr0000gp/T/ipykernel_20570/3676992034.py:8: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  realtor = pd.read_csv(realtor_url)


In [5]:
# we need a univeral metro indexing system - each source uses their own indexing types
# start with case shiller 10 metro areas

cx = {
    'San Francisco': {
        'parcl_id': 2900336,
        'zillow_id': 395057,
        'realtor_id': 41860
    },
    
    'Greater Los Angeles': {
        'parcl_id': 2900078,
        'zillow_id': 753899,
        'realtor_id': 31080
    },
    
    'New York Metropolitan Area': {
        'parcl_id': 2900187,
        'zillow_id': 394913,
        'realtor_id': 35620
    },
    'San Diego County': {
        'parcl_id': 2900332,
        'zillow_id': 395056,
        'realtor_id': 41740
    },
    'Chicago Metropolitan Area': {
        'parcl_id': 2899845,
        'zillow_id': 394463,
        'realtor_id': 16980
    },
    
    'Miami Metropolitan Area': {
        'parcl_id': 2900128,
        'zillow_id': 394856,
        'realtor_id': 33100
    },
    
    'Greater Boston Metropolitan Area': {
        'parcl_id': 2899625,
        'zillow_id': 394404,
        'realtor_id': 14460
    },
    
    'Denver-Aurora Metropolitan Area': {
        'parcl_id': 2899750,
        'zillow_id': 394530,
        'realtor_id': 19740
    }, 
    
    'Las Vegas Metropolitan Area': {
        'parcl_id': 2900049,
        'zillow_id': 394775,
        'realtor_id': 29820
    },
    
    'Washington DC, Metrpolitan Area': {
        'parcl_id': 2900475,
        'zillow_id': 395209,
        'realtor_id': 47900
    }
}


def key_lookup(
    code,
    code_key='zillow_id'
):
    """
    return univeral parcl_id based on source specific identifiers
    """
    for region in cx.keys():
        if code == cx[region][code_key]:
            return cx[region]['parcl_id']
        
# get universal parcl_id which corresponds in this case to the metro areas being referenced
# within each source
realtor['parcl_id'] = realtor['cbsa_code'].apply(lambda x: key_lookup(x, code_key='realtor_id'))
zillow['parcl_id'] = zillow['RegionID'].apply(lambda x: key_lookup(x, code_key='zillow_id'))

In [6]:

# zillow comes in a wide format with dates as columns
# need to melt to longer verison to align with realtor and
# make analysis easier
zillow_melted = pd.melt(
    zillow, 
    id_vars=[
        'RegionID', 
        'SizeRank', 
        'RegionName', 
        'StateName', 
        'RegionType', 
        'parcl_id'
    ], 
    var_name='date', 
    value_name='num_inventory'
)

zillow_melted = zillow_melted.loc[zillow_melted['parcl_id'].notnull()]

In [7]:
# subset data down
realtor_melted = realtor.loc[realtor['parcl_id'].notnull()][['month_date_yyyymm', 'cbsa_code', 'cbsa_title', 'parcl_id', 'total_listing_count']]

In [8]:
def format_r_dates(ex):
    dte = datetime.strptime(str(ex), '%Y%m')
    res = calendar.monthrange(dte.year, dte.month)
    return datetime(dte.year, dte.month, res[1])


# format dates for join
realtor_melted['date'] = realtor_melted['month_date_yyyymm'].apply(lambda x: format_r_dates(x))
zillow_melted['date'] = zillow_melted['date'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d'))

In [9]:
# time to merge
z = zillow_melted[['RegionName', 'parcl_id', 'date', 'num_inventory']]
z = z.rename(columns={'num_inventory': 'zillow_num_inventory'})

r = realtor_melted[['parcl_id', 'total_listing_count', 'date']]
r = r.rename(columns={'total_listing_count': 'realtor_num_inventory'})
final = pd.merge(z, r, on=['parcl_id', 'date'])

In [36]:
def calc_abs_delta(z, r):
    return abs(z-r)

def larger_source(z, r):
    return 'realtor' if r > z else 'zillow'

final['delta'] = final.apply(lambda x: calc_abs_delta(x['zillow_num_inventory'], x['realtor_num_inventory']), axis=1)
final['larger_source'] = final.apply(lambda x: larger_source(x['zillow_num_inventory'], x['realtor_num_inventory']), axis=1)

In [37]:
# write data out
final.to_csv('inventory_discrepancy_analysis.csv', index=False)

In [38]:
# since 2018, how many total discrepant units were there over the 64 month analysis period?

total_discrepant_units = int(final['delta'].sum())
f'{total_discrepant_units:,} units misaligned since 2018 for 10 markets'

'2,228,555 units misaligned since 2018 for 10 markets'

In [39]:
# what is the average monthly diff?

average_monthly_diff = int(final.groupby('date')['delta'].sum().reset_index(name='diff')['diff'].mean())
f'{average_monthly_diff:,} unit discrepancy per month across 10 markets'

'34,821 unit discrepancy per month across 10 markets'

In [40]:
# What is the average diff by market?

final.groupby('RegionName')['delta'].mean().reset_index()

,RegionName,delta
0,"Boston, MA",2771.218750
1,"Chicago, IL",6150.937500
2,"Denver, CO",1216.000000
3,"Las Vegas, NV",3077.171875
4,"Los Angeles, CA",1962.390625
5,"Miami, FL",6992.109375
6,"New York, NY",8104.031250
7,"San Diego, CA",1221.234375
8,"San Francisco, CA",553.343750
9,"Washington, DC",2772.734375


### To do - Redfin Analysis


```python
import gzip
import shutil
import urllib.request

redfin_url = 'https://redfin-public-data.s3.us-west-2.amazonaws.com/redfin_market_tracker/redfin_metro_market_tracker.tsv000.gz'
urllib.request.urlretrieve(redfin_url, 'redfin.gz')

with gzip.open('redfin.gz', 'rb') as f_in:
    with open('redfin.tsv', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
        
redfin = pd.read_csv('redfin.tsv', sep='\t', lineterminator='\r')
```